In [1]:
import os
os.chdir("../")

In [2]:
%pwd

'd:\\web Development using python\\PROJECTS\\kidney_disease_classification'

In [3]:
os.environ['MLFLOW_TRACKING_URI'] = "https://dagshub.com/bhaveshsisodia2/kidney_disease_classification.mlflow"
os.environ['MLFLOW_TRACKING_USERNAME'] = "bhaveshsisodia2"
os.environ['MLFLOW_TRACKING_PASSWORD'] = "d1a15af5489ce5502c813fd8234ea206a71ff52d"

In [4]:
import tensorflow as tf


In [5]:
model = tf.keras.models.load_model(r"D:\web Development using python\PROJECTS\kidney_disease_classification\artifacts\training\model.h5")

In [6]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int

In [7]:
from kidney_disease_classifier.constants import *
from kidney_disease_classifier.utils.common import read_yaml ,create_directories , save_json
from kidney_disease_classifier import logger


In [8]:

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_evaluation_config(self) -> EvaluationConfig:
        eval_config = EvaluationConfig(
            path_of_model = "artifacts/training/model.h5",
            training_data = "artifacts/data_ingestion/kidney-ct-scan-image",
            all_params = self.params,
            mlflow_uri = "https://dagshub.com/bhaveshsisodia2/kidney_disease_classification.mlflow",
            params_image_size = self.params.IMAGE_SIZE,
            params_batch_size = self.params.BATCH_SIZE
        )

        return eval_config

In [9]:
import tensorflow as tf
from pathlib import Path
import mlflow
import mlflow.keras
from urllib.parse import urlparse
from kidney_disease_classifier import logger

c:\Users\HP\anaconda3\envs\cnncls\lib\site-packages\mlflow\utils\requirements_utils.py:12: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [10]:
class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    def train_valid_generator(self):

        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.20
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )





    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        logger.info(f"Loading model from path: {path}")
        model = tf.keras.models.load_model(path)
        logger.info("Model loaded successfully")
        return model

    def evaluate(self):
        logger.info("Starting model evaluation")
        self.train_valid_generator()
        self.model = self.load_model(self.config.path_of_model)
        self.score = self.model.evaluate(self.valid_generator)
        logger.info(f"Evaluation results: {self.score}")
        self.save_score()

    def save_score(self):
        scores ={"loss": self.score[0] , "accuracy":self.score[1]}
        save_json(path =Path("scores.json"), data= scores)

    def log_into_mlflow(self):
        mlflow.set_registry_uri(self.config.mlflow_uri)
        mlflow.set_experiment("Kidney Disease Classification")
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run(run_name="Evaluation") as mlops_run:
            mlflow.log_params(self.config.all_params)
            mlflow.log_metrics({"loss": self.score[0], "accuracy": self.score[1]})

            if tracking_url_type_store != "file":
                mlflow.keras.log_model(self.model, "model", registered_model_name="KidneyDiseaseClassificationModel")
            else:
                mlflow.keras.log_model(self.model, "model")







In [11]:
try:
    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()
    logger.info(f"Evaluation config: {eval_config}")
    evaluation = Evaluation(config=eval_config)
    evaluation.evaluate()
    evaluation.log_into_mlflow()
except Exception as e:
    logger.exception(e)
    raise e

[2026-04-06 13:47:11,625: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-04-06 13:47:11,652: INFO: common: yaml file: params.yaml loaded successfully]
[2026-04-06 13:47:11,655: INFO: common: created directory at: artifacts]
[2026-04-06 13:47:11,659: INFO: 3378058876: Evaluation config: EvaluationConfig(path_of_model='artifacts/training/model.h5', training_data='artifacts/data_ingestion/kidney-ct-scan-image', all_params=ConfigBox({'AUGMENTATION': True, 'IMAGE_SIZE': [224, 224, 3], 'BATCH_SIZE': 16, 'INCLUDE_TOP': False, 'EPOCHS': 2, 'CLASSES': 2, 'WEIGHTS': 'imagenet', 'LEARNING_RATE': 0.02}), mlflow_uri='https://dagshub.com/bhaveshsisodia2/kidney_disease_classification.mlflow', params_image_size=BoxList([224, 224, 3]), params_batch_size=16)]
[2026-04-06 13:47:11,663: INFO: 3522719874: Starting model evaluation]
Found 93 images belonging to 2 classes.
[2026-04-06 13:47:11,750: INFO: 3522719874: Loading model from path: artifacts/training/model.h5]
[2026-04-06 13:

2026/04/06 13:47:36 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


[2026-04-06 13:47:37,371: WARNING: save: Found untraced functions such as _jit_compiled_convolution_op, _jit_compiled_convolution_op, _jit_compiled_convolution_op, _jit_compiled_convolution_op, _jit_compiled_convolution_op while saving (showing 5 of 14). These functions will not be directly callable after loading.]
INFO:tensorflow:Assets written to: C:\Users\HP\AppData\Local\Temp\tmpu504omo_\model\data\model\assets
[2026-04-06 13:47:38,091: INFO: builder_impl: Assets written to: C:\Users\HP\AppData\Local\Temp\tmpu504omo_\model\data\model\assets]


c:\Users\HP\anaconda3\envs\cnncls\lib\site-packages\_distutils_hack\__init__.py:30: UserWarning: Setuptools is replacing distutils. Support for replacing an already imported distutils is deprecated. In the future, this condition will fail. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(
Registered model 'KidneyDiseaseClassificationModel' already exists. Creating a new version of this model...
2026/04/06 13:50:27 INFO mlflow.tracking._model_registry.client: Waiting up to 300 seconds for model version to finish creation.                     Model name: KidneyDiseaseClassificationModel, version 2
Created version '2' of model 'KidneyDiseaseClassificationModel'.
